In [1]:
import thermomatch as match
match.ThermoImpexGenerator.create_local_database_if_not_existent = True

[2024-09-08 15:52:58.682] [jsonio17] [info] Home directory is /home/dmiron


In [2]:
# Set path to `schemas` and `lua` not from the configuration file
match.ThermoImpexGenerator.setResourcesDirectory("../../../../Resources")
# Connect to Arango database
generator = match.ThermoImpexGenerator("http://localhost:8529", "root", "", "ORD_thermochimie12a", True, match.ThermoImpexGenerator.File, True)

[jsonarango] [info] You are connected to: arango 3.9.12


In [3]:
# Import References (the collection name and VertexDataSource schema would be obtained from the script by default)
keys = generator.importFileToDatabase("FormatStructDataFile", "scripts/references.FormatStructDataFile.json", "data/ref_thermochimie.json", "")  #- bug

In [4]:
# List of implemented thermodynamic formats
#print( generator.allFormats() )
generator = match.ThermoImpexGenerator("http://localhost:8529", "root", "", "ORD_thermochimie12a", True, match.ThermoImpexGenerator.File, True) # bug in PhreeqcImportData.cpp line 510 when loading datasourceVertex db - therefore database needs to be reloaded by uncommenting this line

# Execute import from Phreeqc format data
description = [
    match.ImpexRowDescription("SOLUTION_MASTER_SPECIES", "VertexElement", "records", "FormatTableFile", "scripts/elements.FormatTableFile.json"), # element records
    match.ImpexRowDescription("SOLUTION_SPECIES", "VertexSubstance", "records", "FormatKeyValueFile", "scripts/master_aqsubstances.FormatKeyValueFile.json"), # master species substance records
    match.ImpexRowDescription("SECONDARY_MASTER_SPECIES", "VertexSubstance", "records", "FormatKeyValueFile", "scripts/product_aqsubstances.FormatKeyValueFile.json"), # aqueous species substance records
    match.ImpexRowDescription("SECONDARY_MASTER_SPECIES", "VertexReaction", "records_and_links", "FormatKeyValueFile", "scripts/reaction_aqsubstances.FormatKeyValueFile.json"), # aqueous species reaction records and lniks to reactants
    match.ImpexRowDescription("PHASES", "VertexSubstance", "records", "FormatKeyValueFile", "scripts/product_solidsubstances.FormatKeyValueFile.json"), # solid substance records
    match.ImpexRowDescription("PHASES", "VertexReaction", "records_and_links", "FormatKeyValueFile", "scripts/reaction_solidsubstances.FormatKeyValueFile.json"), # solid reaction records and links to reactants
    match.ImpexRowDescription("PHASES", "VertexPhase", "records_and_links", "FormatKeyValueFile", "scripts/phase_solidsubstances.FormatKeyValueFile.json"), # solid phase records and link to substances (endmembers)
    match.ImpexRowDescription("SIT", "VertexPhase", "records", "FormatTableFile", "scripts/phase_sit.FormatTableFile.json"), # aq_sit phase record
    match.ImpexRowDescription("SIT", "VertexMixingModel", "records", "FormatTableFile", "scripts/mixmod_sit.FormatTableFile.json"), # sit mixmod record
    match.ImpexRowDescription("SIT", "VertexInteraction", "records_and_links", "FormatTableFile", "scripts/interaction_sit.FormatTableFile.json"), # interaction records and link to species
    match.ImpexRowDescription("refs", "VertexDataSource", "records_and_links", "FormatStructDataFile", "scripts/references.FormatStructDataFile.json"), # makes links between references and reaction records
    match.ImpexRowDescription("ThermoDataSet", "VertexThermoDataSet", "records_and_links", "FormatTableFile", "scripts/thermodataset.FormatTableFile.json") # ThermoDataSet record and link to elements, substances, rdc
]

[jsonarango] [info] You are connected to: arango 3.9.12


### Preparing file for import
The original PhreeqC_SIT_electron_10a_22Nov2018.dat that can be downloaded from the thermochimie website does not contain `SECONDARY_MASTER_SPECIES` key. For the script to work add:

`SECONDARY_MASTER_SPECIES` after
```
1.000Zr+4     = Zr+4
     log_k     0.000     
     delta_h   0.000     #kJ/mol        
     # Enthalpy of formation:           -608.5        #kJ/mol        #05BRO/CUR
     -analytic 0E+0 0E+0 0E+0 0E+0 0E+0

In [5]:
#print( description ) ~1 min
#generator.runImport("PhreeqcImport", description, "data/PhreeqC_SIT_electron_10a_22Nov2018.dat", "rdc_THERMOCHIMIE10a");
generator.runImport("PhreeqcImport", description, "data/PHREEQC_SIT_e-_ThermoChimie_v12a.dat", "rdc_THERMOCHIMIE12a");

[2024-09-08 15:53:06.597] [thermomatch] [info]  0% Preparing files ...
[2024-09-08 15:53:06.674] [thermomatch] [info]  10% Importing data ...
[2024-09-08 15:53:06.674] [thermomatch] [info]  10% Importing SOLUTION_MASTER_SPECIES VertexElement ...
[jsonio17] [info] VertexElement loading collection: 26, loading query: 1050
[2024-09-08 15:53:06.917] [thermomatch] [info]  15% Importing SOLUTION_SPECIES VertexSubstance ...
[jsonio17] [info] VertexSubstance loading collection: 12, loading query: 797
[2024-09-08 15:53:07.131] [thermomatch] [info]  20% Importing SECONDARY_MASTER_SPECIES VertexSubstance ...
[2024-09-08 15:53:13.721] [thermomatch] [info]  25% Importing SECONDARY_MASTER_SPECIES VertexReaction ...
[jsonio17] [info] VertexReaction loading collection: 11, loading query: 771
[2024-09-08 15:53:20.865] [thermomatch] [info]  30% Importing PHASES VertexSubstance ...
[2024-09-08 15:53:26.736] [thermomatch] [info]  35% Importing PHASES VertexReaction ...
[2024-09-08 15:53:32.999] [thermomat

In [6]:
# update data of elements from CODATA, NEA and SUCPRT
fields_list = [ "_id", "properties.symbol" ]
generator.updateFileToDatabase("FormatTableFile", "scripts/elements_update.FormatTableFile.json",
                    "../elements/data/elements.csv", "", "", fields_list )